# 00 — Antioquia (Medellín-department) SISAIRE network, per-variable QC

`Data/COLOMBIA/ANTIOQUIA_*_reporte_sisaire.csv` is the same per-station daily-mean
SISAIRE export format Bogotá's own pipeline uses (`Estacion,Fecha inicial,Fecha
final,<value>`), previously missed for Medellín -- the pipeline had instead used a
pre-aggregated `Hourmean` network-mean series (no visibility into which/how many
stations feed it) for PM2.5/PM10/NO2/O3, and only a small 3-station minute-resolution
export (`DATOS_MEDELLIN`, notebook 01b) for meteorology, capped at 2020.

The Antioquia files cover **far more stations** (47-216, vs. the unknown handful behind
`Hourmean`) and **much longer date ranges**: meteorology (wind/pressure/10m-temp/10m-RH)
extends to **2026** (current pipeline: ground-truthed only to 2020), and PM10/PM2.5/NO2
go back to 1998-2008 (current pipeline: whatever `Hourmean` covers, undocumented
provenance).

**O3 is deliberately NOT rebuilt from this source.** `ANTIOQUIA_O3` is a plain
daily-mean export (same convention as every other file here), not MDA8 -- computing
MDA8 needs the underlying hourly data, which isn't available pre-2019 in any source
found. Mixing MDA8 (2019+, from `Hourmean`'s hourly data) with plain daily-mean
(pre-2019, from this file) into one `o3` column would silently blend two different
metrics. Pre-2019 station coverage is also patchy (many station-years likely fail the
completeness threshold below) for a partial extension that wouldn't obviously be worth
the metric inconsistency. O3 stays exactly as built in `01_load_medellin_network.ipynb`.

**QC approach**: unlike Bogotá's `02_quality_control.ipynb` (one completeness gate on
`pm25`, applied to every variable since Bogotá's ~20 stations report all variables
together), each variable here gets its **own independent** Chauvenet + >=75%
completeness station-year selection -- appropriate given how heterogeneous per-station-
variable coverage is across a 47-216-station network (a station reporting PM2.5
completely may report NO2 sparsely, or not at all).

**Geographic filter added after initial run**: the raw files cover the WHOLE Antioquia
department (303 unique stations across 8 variables), not just the Aburrá Valley/Medellín
metro area -- confirmed by a real bug this exposed: the unfiltered `pressure` network
mean averaged in high-altitude/distant stations (Oriente Antioqueño, Bajo Cauca, etc.),
producing a network mean of ~627 hPa vs. ERA5's ~803 hPa (corr=0.11 -- clearly wrong).
Filtered to the 10 official Valle de Aburrá municipalities (Medellín, Bello, Itagüí,
Envigado, Sabaneta, La Estrella, Caldas, Copacabana, Girardota, Barbosa) by matching
station names against the municipality name or a known AMVA/SIATA short-code prefix
(`MED-`, `ITA-`, `ENV-`, `GIR-`, `BEL-`, `SAB-`, `CAL-`, `COP-`, `BAR-`), plus two
explicit additions for stations that don't spell out a municipality but are
independently confirmed local (Universidad Nacional's "Núcleo El Volador" campus is in
Medellín; Corporación Universitaria Lasallista is headquartered in Caldas, Antioquia).
This is a name-based heuristic, not a coordinate lookup (no station coordinate catalog
exists for this dataset either) -- some genuinely Aburrá stations with ambiguous, code-
only names (e.g. "DITAIRES", "TRES RANCHOS", "ESTACION SAN ANTONIO MONITOREO CONTINUO")
are conservatively excluded rather than guessed, a real but bounded limitation.

**Pressure dropped after inspection**: per-station means for `pressure` ranged from
551-872 hPa across the (Aburra-filtered) network -- only 4 of 52 stations report
physically plausible values for Medellin's ~1495m altitude (~800-870 hPa); the rest
cluster 551-750 hPa, consistent with widespread barometric-sensor miscalibration (common
in air-quality stations where pressure isn't the primary measurement). This is a
per-station systematic bias, not the kind of within-station outlier Chauvenet's
criterion catches, and no station-coordinate catalog exists to build a principled
good-sensor filter beyond "does this station's own mean look physically plausible."
Dropped entirely; ERA5's pressure (already used throughout this project) remains the
sole source. `no2`'s weaker correlation with the existing `Hourmean`-sourced series
(checked after building the merged file) reflects a much wider, traffic-station-inclusive
network here vs. `Hourmean`'s narrower undocumented set -- a real network-representativeness
difference (confirmed via the smooth traffic-proximity gradient in per-station means, not
a bimodal miscalibration pattern like pressure's), not a data-quality bug -- kept, but
flagged for the calibration step in the master-dataset merge.


In [1]:
import sys
from pathlib import Path
import unicodedata

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.sisaire_io import load_sisaire_file
from common.qc import (
    apply_physical_range_checks, flag_chauvenet_outliers,
    completeness_by_station_year, select_stations_years, PHYSICAL_RANGES,
)

RAW_DIR = PROJECT_ROOT / "Data" / "COLOMBIA"
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"

# (tidy_name, exact filename) -- two files have an irregular naming pattern (no
# underscore before "reporte"), so paths are given explicitly rather than built from a
# f"{prefix}_{code}_reporte_sisaire.csv" pattern.
FILES = {
    "pm25": "ANTIOQUIA_PM25_reporte_sisaire.csv",
    "pm10": "ANTIOQUIA_PM10_reporte_sisaire.csv",
    "no2": "ANTIOQUIA_NO2_reporte_sisaire.csv",
    "wind_speed": "ANTIOQUIA_VV_reporte_sisaire.csv",
    "wind_dir": "ANTIOQUIA_DV_reporte_sisaire.csv",
    "temp_10m": "ANTIOQUIA_TAire10_reporte_sisaire.csv",
    "rh_10m": "ANTIOQUIA_HAire10_reporte_sisaire.csv",
}

# Geographic filter: the raw files cover the WHOLE Antioquia department (303 unique
# stations across these 8 files), not just the Aburra Valley/Medellin metro area. See
# markdown above for the bug this caused (unfiltered pressure network mean ~627 hPa vs.
# ERA5's ~803 hPa) and the reasoning behind this name-based whitelist.
def _strip_accents(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if unicodedata.category(ch) != 'Mn')

ABURRA_MUNICIPALITIES = ['medellin', 'itagui', 'envigado', 'sabaneta', 'estrella', 'caldas',
                          'copacabana', 'girardota', 'barbosa', 'bello']
ABURRA_PREFIX_CODES = ['MED-', 'ITA-', 'ENV-', 'GIR-', 'BEL-', 'SAB-', 'CAL-', 'COP-', 'BAR-']
ABURRA_EXPLICIT_EXTRA = ['UNIVERSIDAD NACIONAL - N\u00daCLEO EL VOLADOR', 'CORPORACI\u00d3N UNIVERSITARIA LASALLISTA']

def is_aburra_station(name: str) -> bool:
    norm = _strip_accents(name).lower()
    if any(m in norm for m in ABURRA_MUNICIPALITIES):
        return True
    if any(name.upper().startswith(p) for p in ABURRA_PREFIX_CODES):
        return True
    if name.strip().upper() in ABURRA_EXPLICIT_EXTRA:
        return True
    return False

raw = {}
for tidy_name, fname in FILES.items():
    d = load_sisaire_file(RAW_DIR / fname, tidy_name)
    n_before = d["station"].nunique()
    d = d[d["station"].map(is_aburra_station)].reset_index(drop=True)
    n_after = d["station"].nunique()
    raw[tidy_name] = d
    dmin, dmax = d['date'].min(), d['date'].max()
    print(f"{tidy_name}: {n_before} -> {n_after} stations after Aburra filter  "
          f"{d.shape}  {dmin} to {dmax}")


pm25: 141 -> 54 stations after Aburra filter  (76037, 3)  2008-09-20 00:00:00 to 2026-08-16 00:00:00


pm10: 202 -> 43 stations after Aburra filter  (54918, 3)  2003-01-13 00:00:00 to 2026-08-16 00:00:00
no2: 47 -> 19 stations after Aburra filter  (22720, 3)  2001-05-11 00:00:00 to 2026-08-16 00:00:00


wind_speed: 216 -> 55 stations after Aburra filter  (55692, 3)  2009-07-18 00:00:00 to 2026-08-16 00:00:00


wind_dir: 141 -> 46 stations after Aburra filter  (54782, 3)  2011-11-03 00:00:00 to 2026-08-16 00:00:00
temp_10m: 70 -> 28 stations after Aburra filter  (13896, 3)  2013-02-01 00:00:00 to 2026-08-16 00:00:00


rh_10m: 90 -> 40 stations after Aburra filter  (49978, 3)  2013-01-01 00:00:00 to 2026-08-16 00:00:00


## Per-variable QC: physical range + Chauvenet + >=75% annual completeness

In [2]:
RANGES = {**PHYSICAL_RANGES}  # pm25/pm10/no2/wind_speed/wind_dir/temp_10m/rh_10m already defined

network_series = {}
completeness_tables = {}
for tidy_name, d in raw.items():
    clean = apply_physical_range_checks(d.rename(columns={tidy_name: tidy_name}), {tidy_name: RANGES.get(tidy_name, PHYSICAL_RANGES.get(tidy_name, (-1e9, 1e9)))})
    clean = flag_chauvenet_outliers(clean, [tidy_name], group_col="station")

    completeness = completeness_by_station_year(clean, tidy_name, date_col="date", station_col="station")
    qualifying = select_stations_years(completeness, threshold=0.75)
    completeness_tables[tidy_name] = completeness

    clean["year"] = clean["date"].dt.year
    qualifying_keys = set(zip(qualifying["station"], qualifying["year"]))
    clean["is_qualifying"] = clean.apply(lambda r: (r["station"], r["year"]) in qualifying_keys, axis=1)
    qc_rows = clean[clean["is_qualifying"]]

    network = qc_rows.groupby("date")[tidy_name].mean().reset_index()
    network_series[tidy_name] = network
    print(f"{tidy_name}: {len(qualifying)}/{len(completeness)} station-years qualify (>=75%) -- "
          f"network series {network.shape}, {network['date'].min()} to {network['date'].max()}")


[Chauvenet] pm25: flagged 735 outliers


pm25: 166/282 station-years qualify (>=75%) -- network series (4381, 2), 2009-01-01 00:00:00 to 2025-12-31 00:00:00
[Chauvenet] pm10: flagged 286 outliers
pm10: 87/323 station-years qualify (>=75%) -- network series (5299, 2), 2008-01-01 00:00:00 to 2025-12-31 00:00:00
[Chauvenet] no2: flagged 67 outliers


no2: 51/93 station-years qualify (>=75%) -- network series (4207, 2), 2011-01-18 00:00:00 to 2025-12-31 00:00:00
[range check] wind_speed: 61 values outside [0.0, 30.0] -> NaN
[Chauvenet] wind_speed: flagged 303 outliers
wind_speed: 120/219 station-years qualify (>=75%) -- network series (4287, 2), 2010-01-01 00:00:00 to 2025-12-31 00:00:00
[Chauvenet] wind_dir: flagged 258 outliers


wind_dir: 119/204 station-years qualify (>=75%) -- network series (3986, 2), 2013-01-01 00:00:00 to 2025-12-31 00:00:00
[range check] temp_10m: 58 values outside [-5.0, 35.0] -> NaN
[Chauvenet] temp_10m: flagged 31 outliers
temp_10m: 18/64 station-years qualify (>=75%) -- network series (1735, 2), 2013-04-01 00:00:00 to 2023-12-31 00:00:00
[Chauvenet] rh_10m: flagged 446 outliers
rh_10m: 108/186 station-years qualify (>=75%) -- network series (3551, 2), 2013-04-01 00:00:00 to 2025-12-31 00:00:00


## Merge all variables into one wide daily table

In [3]:
from functools import reduce

network_daily = reduce(
    lambda a, b: a.merge(b, on="date", how="outer"),
    network_series.values(),
)
network_daily = network_daily.sort_values("date").reset_index(drop=True)
print(network_daily.shape)
network_daily.describe()


(6081, 8)


,date,pm25,pm10,no2,wind_speed,wind_dir,temp_10m,rh_10m
count,6081,4371.000000,5299.000000,4207.000000,4286.000000,3986.000000,1735.000000,3551.000000
mean,2017-02-12 20:05:33.892452,19.818600,51.699813,29.845703,1.308368,149.320366,21.444010,73.448559
min,2008-01-01 00:00:00,0.000000,14.965529,0.002950,0.000000,0.000000,16.234375,0.000000
25%,2012-05-04 00:00:00,14.356067,37.505910,20.251780,1.040020,135.752626,20.133074,68.659031
50%,2017-08-17 00:00:00,17.900000,46.449939,31.940668,1.285535,148.121778,21.558333,74.454127
75%,2021-11-02 00:00:00,23.772895,59.562307,40.172822,1.544972,161.519265,22.757292,79.435198
max,2025-12-31 00:00:00,55.187500,393.000000,79.029167,3.523904,337.500000,27.666667,91.333422
std,NaN,7.686761,24.157468,16.767802,0.383398,25.068289,1.823301,8.449158


## Sanity check: physical plausibility + coverage by year

In [4]:
network_daily["year"] = network_daily["date"].dt.year
for col in ["pm25", "pm10", "no2", "wind_speed", "wind_dir", "temp_10m", "rh_10m"]:
    cov = network_daily.groupby("year")[col].apply(lambda s: s.notna().mean())
    nonzero_years = cov[cov > 0]
    print(f"--- {col}: non-null years {nonzero_years.index.min()}-{nonzero_years.index.max()}, "
          f"mean coverage in those years {nonzero_years.mean():.2f} ---")
network_daily = network_daily.drop(columns=["year"])


--- pm25: non-null years 2009-2025, mean coverage in those years 0.93 ---
--- pm10: non-null years 2008-2025, mean coverage in those years 0.98 ---
--- no2: non-null years 2011-2025, mean coverage in those years 0.96 ---
--- wind_speed: non-null years 2010-2025, mean coverage in those years 0.98 ---
--- wind_dir: non-null years 2013-2025, mean coverage in those years 0.99 ---
--- temp_10m: non-null years 2013-2023, mean coverage in those years 0.95 ---
--- rh_10m: non-null years 2013-2025, mean coverage in those years 0.97 ---


## Save

In [5]:
out_path = OUT_DIR / "medellin_antioquia_network_daily.parquet"
network_daily.to_parquet(out_path, index=False)
print("Saved:", out_path, network_daily.shape)

for tidy_name, comp in completeness_tables.items():
    comp.to_csv(OUT_DIR / f"medellin_antioquia_{tidy_name}_completeness_by_station_year.csv", index=False)
print("Saved completeness tables for", list(completeness_tables.keys()))


Saved: /path/to/project/Data/processed/colombia/medellin/medellin_antioquia_network_daily.parquet (6081, 8)
Saved completeness tables for ['pm25', 'pm10', 'no2', 'wind_speed', 'wind_dir', 'temp_10m', 'rh_10m']
